# 2. Trip descriptors across many VED trips

This notebook computes standard trip descriptors (average speed, idle share, number of stops, vehicle-specific power, and so on) across a batch of real VED trips, and looks at the resulting distributions.

In [1]:
VED_CSV_PATH = "VED_171101_week.csv"  # change to your extracted file
N_TRIPS = 100  # how many trips to process, for a quick example

In [2]:
from drivecycle_stats.descriptors import descriptors_table
from drivecycle_stats.io_ved import clean_trip, iter_ved_trips, load_ved_dynamic_csv

raw = load_ved_dynamic_csv(VED_CSV_PATH)

frames = []
ids = []
for veh_id, trip_id, raw_trip in iter_ved_trips(raw):
    frame, report = clean_trip(raw_trip)
    if frame is not None:
        frames.append(frame)
        ids.append(f"{veh_id}_{trip_id}")
    if len(frames) >= N_TRIPS:
        break

print(f"{len(frames)} trips cleaned and kept")

100 trips cleaned and kept


In [3]:
table = descriptors_table(frames, ids=ids)
table.describe()

,distance_km,duration_s,avgspd,runspd,avgposacc,rmsa,idle_share,n_stops,mean_stop_duration_s,v95,vsp_pos_mean
count,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,87.000000,100.000000,100.000000
mean,4.339969,505.970000,37.550494,45.096699,0.666507,0.719184,0.181863,3.960000,27.125084,67.674138,6.756653
std,3.077576,870.445439,14.059021,12.575433,0.136539,0.131721,0.145921,4.681621,28.089389,17.676733,2.329518
min,0.631637,82.000000,3.805082,19.213910,0.390113,0.258754,0.000000,0.000000,1.000000,24.093939,2.356231
25%,1.780659,204.000000,27.393063,38.110041,0.561015,0.633228,0.065401,1.000000,14.250000,61.000000,5.467213
50%,3.495062,357.500000,35.632714,44.087707,0.656708,0.722777,0.172184,2.500000,20.500000,65.000000,6.352783
75%,5.821233,589.500000,44.660322,50.231960,0.749771,0.788507,0.262672,5.250000,31.687500,73.071250,7.660110
max,14.389405,8647.000000,82.378488,86.978861,1.032637,1.069401,0.802012,30.000000,231.166667,130.730000,15.519202


Flag unusual trips using the low-density outlier check on two descriptors.

In [4]:
from drivecycle_stats.outliers import flag_low_density

xy = table[["avgspd", "idle_share"]].dropna().to_numpy()
flags = flag_low_density(xy, pctl=5.0)
print(f"{flags.sum()} of {len(xy)} trips flagged as low-density outliers at pctl=5")

5 of 100 trips flagged as low-density outliers at pctl=5
